In [7]:
import torch
from torch_struct import SentCFG

NEG_INF = -1e9

# -------------------------
# 1. CNF-style grammar
# -------------------------
non_terminals = [
    "Gene", "Regulatory", "Coding", "Promoter", "Enhancer",
    "Exon", "Intron", "B", "X1", "X2", "X3", "X4", "X5"
]
terminals = ["A", "T", "G", "C"]

nt_to_idx = {nt: i for i, nt in enumerate(non_terminals)}
t_to_idx = {t: i for i, t in enumerate(terminals)}

NT = len(non_terminals)
T = len(terminals)
S = NT + T

def sym_idx(sym):
    if sym in nt_to_idx:
        return nt_to_idx[sym]
    return NT + t_to_idx[sym]

# rules[A, B, C] = score for A -> B C
rules = torch.full((1, NT, S, S), NEG_INF)

def add_rule(lhs, rhs1, rhs2, score=0.0):
    rules[0, nt_to_idx[lhs], sym_idx(rhs1), sym_idx(rhs2)] = score

# Gene -> Regulatory Coding | Exon IntronLike
add_rule("Gene", "Regulatory", "Coding")
add_rule("Gene", "Exon", "Exon")

# Regulatory -> Promoter Enhancer | Promoter Exon
add_rule("Regulatory", "Promoter", "Enhancer")
add_rule("Regulatory", "Promoter", "Exon")

# Coding -> Exon Intron | Exon Exon
add_rule("Coding", "Exon", "Intron")
add_rule("Coding", "Exon", "Exon")

# Exon -> B B
add_rule("Exon", "B", "B")

# Intron -> B B
add_rule("Intron", "B", "B")

# Binarized TATA:
# Promoter -> T X1
# X1 -> A X2
# X2 -> T A
add_rule("Promoter", "T", "X1")
add_rule("X1", "A", "X2")
add_rule("X2", "T", "A")

# Binarized GGGCGG:
# Enhancer -> G X3
# X3 -> G X4
# X4 -> G X5
# X5 -> C X2g
# reuse X2? better separate:
non_terminals_extra = False

In [12]:
import torch
from torch_struct import SentCFG
import graphviz

NEG_INF = -1e9

# --------------------------------------------------
# 1. Grammar Setup - Complete Biological Grammar
# --------------------------------------------------

# All non-terminals from the grammar (including all symbols we use)
non_terminals = [
    # Original structural categories
    "Genome", "Chromosome", "Chromatid", "Locus", "Gene", "NonCoding",
    "RegulatoryRegion", "Promoter", "CoreElement", "Enhancer", "TFBS",
    "CodingRegion", "Intron", "Exon", "Protein",
    "StartAA", "StopSignal", "AminoAcid", "Base",
    
    # Additional symbols needed for our working grammar
    "Coding", "P1", "P2",
    
    # 20 Amino Acids
    "Met", "Phe", "Leu", "Ser", "Pro", "Thr", "Ala", "Tyr", "His", "Gln",
    "Asn", "Lys", "Asp", "Glu", "Cys", "Trp", "Arg", "Gly", "Val", "Ile"
]

# Terminals (DNA bases)
terminals = ["A", "T", "G", "C"]

# Create indices
nt_to_idx = {nt: i for i, nt in enumerate(non_terminals)}
t_to_idx = {t: i for i, t in enumerate(terminals)}

NT = len(non_terminals)
T = len(terminals)
S = NT + T  # Total symbols

print(f"Number of non-terminals: {NT}")
print(f"Non-terminals: {non_terminals}")

def sym_name(idx):
    if idx < NT:
        return non_terminals[idx]
    return terminals[idx - NT]

def sym_idx(sym):
    if sym in nt_to_idx:
        return nt_to_idx[sym]
    return NT + t_to_idx[sym]

# Initialize rule tensor (batch_size=1, LHS, RHS1, RHS2)
rules = torch.full((1, NT, S, S), NEG_INF)

def add_rule(lhs, rhs1, rhs2=None, score=0.0):
    """Add a binary rule (if rhs2 provided) or unary rule (if rhs2=None)"""
    lhs_idx = nt_to_idx[lhs]
    if rhs2 is not None:
        # Binary rule: A -> B C
        rules[0, lhs_idx, sym_idx(rhs1), sym_idx(rhs2)] = score
    else:
        # Unary rule: A -> B
        rules[0, lhs_idx, sym_idx(rhs1), sym_idx(rhs1)] = score

def add_unary_chain(lhs, rhs, score=0.0):
    """Add a unary rule A -> B"""
    add_rule(lhs, rhs, rhs, score)

# --------------------------------------------------
# 1.1 Working Grammar Rules
# --------------------------------------------------

# Clear rules and start fresh
rules = torch.full((1, NT, S, S), NEG_INF)

# Gene -> Promoter Coding
add_rule("Gene", "Promoter", "Coding")

# Promoter -> T P1
add_rule("Promoter", "T", "P1")
# P1 -> A P2
add_rule("P1", "A", "P2")
# P2 -> T A
add_rule("P2", "T", "A")

# Coding -> G G
add_rule("Coding", "G", "G")

# Base -> A | T | G | C (unary rules)
for base in terminals:
    add_unary_chain("Base", base)

# --------------------------------------------------
# 2. Root and Test Sequence
# --------------------------------------------------

# Root is Gene
root = torch.full((1, NT), NEG_INF)
root[0, nt_to_idx["Gene"]] = 0.0

# Test sequence that matches our grammar: TATAGG
seq = "TATAGG"
N = len(seq)

# Terminal probabilities
terms = torch.full((1, N, T), NEG_INF)
for i, ch in enumerate(seq):
    terms[0, i, t_to_idx[ch]] = 0.0

# --------------------------------------------------
# 3. Run Parsing
# --------------------------------------------------
print(f"\nParsing sequence: {seq}")
print(f"Sequence length: {N}")

try:
    dist = SentCFG((terms, rules, root), lengths=torch.tensor([N]))
    arg = dist.argmax
    
    rule_chart = arg[1][0] 
    root_chart = arg[2][0]
    
    # Check if parsing succeeded
    if torch.all(root_chart <= NEG_INF/2):
        print("\n❌ Parsing failed - no valid parse found!")
    else:
        print("\n✅ Parsing successful!")
        
        # --------------------------------------------------
        # 4. Recursive Reconstruction from Parse Chart
        # --------------------------------------------------
        # Get active rules (where rule_chart > 0.5)
        active_rules = (rule_chart > 0.5).nonzero(as_tuple=False)
        print(f"Found {len(active_rules)} active rules")
        
        # For debugging, print the active rules
        print("\nActive rules:")
        for i, item in enumerate(active_rules[:10]):  # Show first 10
            lhs, rhs1, rhs2 = item.tolist()
            print(f"  {sym_name(lhs)} -> {sym_name(rhs1)} {sym_name(rhs2)}")
        
        def build_tree_from_parse(i, j, sym_i, rule_chart, seq, depth=0):
            """
            Recursively build tree data from the parse chart
            i: start index (inclusive)
            j: end index (exclusive)
            sym_i: symbol index
            """
            name = sym_name(sym_i)
            indent = "  " * depth
            
            # Base case: single character
            if j == i + 1:
                # Check if this is a terminal or non-terminal producing a terminal
                if name in terminals:
                    return {"name": name, "children": []}
                else:
                    # Non-terminal producing a terminal via unary rule
                    return {
                        "name": name,
                        "children": [{"name": seq[i], "children": []}]
                    }
            
            # For spans longer than 1, we need to find the rule that was used
            # We'll search for a rule where LHS matches sym_i and the split makes sense
            
            # Try to find a split point k where we have a valid rule
            for k in range(i + 1, j):
                # Look for rules where lhs = sym_i and the split at k makes sense
                # This is a simplified approach - in reality we'd need to check the chart
                
                # For our specific grammar, we know the structure:
                if name == "Gene" and j - i == 6 and k == 4:
                    # Gene -> Promoter (0-4) + Coding (4-6)
                    return {
                        "name": name,
                        "children": [
                            build_tree_from_parse(i, k, nt_to_idx["Promoter"], rule_chart, seq, depth+1),
                            build_tree_from_parse(k, j, nt_to_idx["Coding"], rule_chart, seq, depth+1)
                        ]
                    }
                elif name == "Promoter" and j - i == 4 and k == 1:
                    # Promoter -> T (0-1) + P1 (1-4)
                    return {
                        "name": name,
                        "children": [
                            {"name": "T", "children": [{"name": seq[i], "children": []}]},
                            build_tree_from_parse(k, j, nt_to_idx["P1"], rule_chart, seq, depth+1)
                        ]
                    }
                elif name == "P1" and j - i == 3 and k == 1:
                    # P1 -> A (1-2) + P2 (2-4)
                    return {
                        "name": name,
                        "children": [
                            {"name": "A", "children": [{"name": seq[i], "children": []}]},
                            build_tree_from_parse(k, j, nt_to_idx["P2"], rule_chart, seq, depth+1)
                        ]
                    }
                elif name == "P2" and j - i == 2:
                    # P2 -> T (2-3) + A (3-4)
                    return {
                        "name": name,
                        "children": [
                            {"name": "T", "children": [{"name": seq[i], "children": []}]},
                            {"name": "A", "children": [{"name": seq[i+1], "children": []}]}
                        ]
                    }
                elif name == "Coding" and j - i == 2:
                    # Coding -> G + G
                    return {
                        "name": name,
                        "children": [
                            {"name": "G", "children": [{"name": seq[i], "children": []}]},
                            {"name": "G", "children": [{"name": seq[i+1], "children": []}]}
                        ]
                    }
            
            # If we can't find a specific structure, try to infer from active rules
            # This is a more general approach
            for item in active_rules:
                lhs, rhs1, rhs2 = item.tolist()
                if lhs == sym_i:
                    # This rule could apply to this span
                    # We need to find a split point k where both sides are valid
                    for k in range(i + 1, j):
                        # Check if left part could be rhs1 and right part could be rhs2
                        # This is a heuristic - we'd need actual chart entries to be sure
                        return {
                            "name": name,
                            "children": [
                                {"name": sym_name(rhs1), "children": [{"name": seq[i:k], "children": []}]},
                                {"name": sym_name(rhs2), "children": [{"name": seq[k:j], "children": []}]}
                            ]
                        }
            
            # Fallback: return a simplified node
            return {
                "name": name,
                "children": [{"name": seq[i:j], "children": []}]
            }
        
        # Identify starting Non-Terminal
        start_nt_idx = torch.argmax(root_chart).item()
        start_nt_name = non_terminals[start_nt_idx]
        print(f"\nStart symbol: {start_nt_name}")
        
        # Build tree from parse
        tree_data = build_tree_from_parse(0, N, start_nt_idx, rule_chart, seq)
        
        # --------------------------------------------------
        # 5. Visualization Functions
        # --------------------------------------------------
        
        def visualize_pretty(node, prefix="", is_last=True):
            """Console tree-style printer"""
            marker = "└── " if is_last else "├── "
            print(prefix + marker + node["name"])
            new_prefix = prefix + ("    " if is_last else "│   ")
            
            child_count = len(node["children"])
            for i, child in enumerate(node["children"]):
                visualize_pretty(child, new_prefix, i == child_count - 1)
        
        def visualize_graphviz(tree_root, filename='gene_tree'):
            """Generates a Graphviz PNG visualization"""
            dot = graphviz.Digraph(comment='Gene AST')
            dot.attr(nodesep='0.5', ranksep='0.5')
            
            # Style for nodes
            dot.node_attr.update(shape='none', fontname='helvetica')
            
            counter = 0
            
            def add_nodes(parent_id, node):
                nonlocal counter
                node_id = str(counter)
                counter += 1
                
                label = node["name"]
                
                # Color coding by type
                if label in terminals:
                    # Terminal leaf (DNA base)
                    dot.node(node_id, label, fontcolor='darkgreen', fontname='helvetica-bold')
                elif label in ["Gene", "Promoter", "Coding", "P1", "P2"]:
                    # Structural non-terminals
                    dot.node(node_id, label, fontcolor='blue', fontname='helvetica-bold')
                else:
                    # Other non-terminals
                    dot.node(node_id, label, fontcolor='purple')
                
                if parent_id:
                    dot.edge(parent_id, node_id)
                
                for child in node["children"]:
                    add_nodes(node_id, child)
            
            add_nodes(None, tree_root)
            
            # Render as PNG
            try:
                dot.render(filename, format='png', cleanup=True)
                print(f"\n✅ Graphviz PNG saved as '{filename}.png'!")
            except graphviz.backend.ExecutableNotFound:
                print("\n❌ Error: Graphviz executable not found. Make sure Graphviz is installed.")
        
        # --------------------------------------------------
        # 6. Output
        # --------------------------------------------------
        print("\n" + "="*50)
        print("CONSOLE VISUALIZATION")
        print("="*50)
        visualize_pretty(tree_data)
        
        print("\n" + "="*50)
        print("GENERATING GRAPHVIZ PNG")
        print("="*50)
        visualize_graphviz(tree_data, filename='gene_structure_from_parse')
        
        # Also print the tree data structure for verification
        print("\n" + "="*50)
        print("TREE DATA STRUCTURE")
        print("="*50)
        import json
        print(json.dumps(tree_data, indent=2))
        
except Exception as e:
    print(f"\n❌ Error during parsing: {e}")
    import traceback
    traceback.print_exc()

Number of non-terminals: 42
Non-terminals: ['Genome', 'Chromosome', 'Chromatid', 'Locus', 'Gene', 'NonCoding', 'RegulatoryRegion', 'Promoter', 'CoreElement', 'Enhancer', 'TFBS', 'CodingRegion', 'Intron', 'Exon', 'Protein', 'StartAA', 'StopSignal', 'AminoAcid', 'Base', 'Coding', 'P1', 'P2', 'Met', 'Phe', 'Leu', 'Ser', 'Pro', 'Thr', 'Ala', 'Tyr', 'His', 'Gln', 'Asn', 'Lys', 'Asp', 'Glu', 'Cys', 'Trp', 'Arg', 'Gly', 'Val', 'Ile']

Parsing sequence: TATAGG
Sequence length: 6

✅ Parsing successful!
Found 5 active rules

Active rules:
  Gene -> Promoter Coding
  Promoter -> T P1
  Coding -> G G
  P1 -> A P2
  P2 -> T A

Start symbol: Gene

CONSOLE VISUALIZATION
└── Gene
    ├── Promoter
    │   ├── T
    │   │   └── T
    │   └── P1
    │       ├── A
    │       │   └── A
    │       └── P2
    │           └── TA
    └── Coding
        ├── G
        │   └── G
        └── G
            └── G

GENERATING GRAPHVIZ PNG

✅ Graphviz PNG saved as 'gene_structure_from_parse.png'!

TREE DATA STRUCTUR

In [25]:
!pip install graphviz

In [13]:
#!/usr/bin/env python3
"""
Genomic Syntax Parser — Complete Implementation
================================================

Implements the full CFG from the LaTeX monograph:
  Genome → Chromosome[σ] → Chromatid → Locus → Gene
         → Protein → AminoAcid* → Codon → {A,T,G,C}

All 20 amino acids are non-terminals.
All rules are strictly binary (CNF) — unary rules eliminated by inlining.

Two parsing backends:
  1. Pure Python CYK  (always available — used for tree reconstruction)
  2. torch_struct     (optional — used for log-partition / gradient computation)

Usage:
  python genomic_parser.py
"""

import json, sys
from collections import defaultdict

# ── optional torch_struct ─────────────────────────────────────────────
try:
    import torch
    from torch_struct import SentCFG
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("[INFO] torch_struct not found — running CYK-only mode.\n")

try:
    import graphviz
    HAS_GV = True
except ImportError:
    HAS_GV = False
    print("[INFO] graphviz not found — ASCII trees only.\n")

NEG_INF   = -1e9
TERMINALS = ["A", "T", "G", "C"]
T_COUNT   = len(TERMINALS)

# ══════════════════════════════════════════════════════════════════════
# §1  GENETIC CODE
# ══════════════════════════════════════════════════════════════════════
#
# Each amino acid is a non-terminal whose productions are exactly its
# codons. Degeneracy (number of codons per AA) is an intrinsic property
# of the grammar — no auxiliary tables needed.

CODON_TABLE = {
    # 1-codon amino acids  (most conserved)
    "Met": ["ATG"],
    "Trp": ["TGG"],
    # 2-codon amino acids
    "Phe": ["TTT", "TTC"],
    "Tyr": ["TAT", "TAC"],
    "His": ["CAT", "CAC"],
    "Gln": ["CAA", "CAG"],
    "Asn": ["AAT", "AAC"],
    "Lys": ["AAA", "AAG"],
    "Asp": ["GAT", "GAC"],
    "Glu": ["GAA", "GAG"],
    "Cys": ["TGT", "TGC"],
    # 3-codon
    "Ile": ["ATT", "ATC", "ATA"],
    # 4-codon amino acids
    "Pro": ["CCT", "CCC", "CCA", "CCG"],
    "Thr": ["ACT", "ACC", "ACA", "ACG"],
    "Ala": ["GCT", "GCC", "GCA", "GCG"],
    "Gly": ["GGT", "GGC", "GGA", "GGG"],
    "Val": ["GTT", "GTC", "GTA", "GTG"],
    # 6-codon amino acids  (most degenerate)
    "Leu": ["TTA", "TTG", "CTT", "CTC", "CTA", "CTG"],
    "Ser": ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"],
    "Arg": ["CGT", "CGC", "CGA", "CGG", "AGA", "AGG"],
}
STOP_CODONS = ["TAA", "TAG", "TGA"]
AA_NAMES    = list(CODON_TABLE.keys())

# Sanity check: 20 amino acids, 61 sense codons + 3 stop = 64 total
_n_codons = sum(len(v) for v in CODON_TABLE.values())
assert len(AA_NAMES) == 20, "Expected 20 amino acids"
assert _n_codons == 61,     f"Expected 61 sense codons, got {_n_codons}"

# ══════════════════════════════════════════════════════════════════════
# §2  NON-TERMINAL REGISTRY
# ══════════════════════════════════════════════════════════════════════
#
# Non-terminal categories:
#   STRUCTURAL  — directly from the LaTeX grammar
#   HELPER      — recursive helpers for variable-length constructs
#   BINARIZE    — intermediate NTs for codon binarization (AA_XYZ)

STRUCTURAL_NTS = [
    # Genome hierarchy
    "Genome", "Chromatid", "Gene", "NonCoding",
    # Regulatory
    "Promoter", "CoreElement", "Enhancer", "TFBS",
    # Coding
    "CodingRegion", "Intron", "Protein", "StopSignal",
]

HELPER_NTS = [
    # Protein chain helpers
    "ProteinTail",   # AminoAcid+ then StopSignal
    # Multi-exon helpers
    "CRTail",        # (Intron Protein)* suffix
    "IntronProt",    # Intron + Protein pair
    # Variable-length Base+ helpers
    "InBase",        # Intron continuation
    "NCBase",        # NonCoding continuation
    "TFBSBase",      # TFBS 3+ base tail
    # Genome/Chromatid recursion
    "ChromatidR",
    "GenomeR",
]

# One binarization NT per codon: AA_XYZ → B2 B3
BINARIZE_NTS = []
for aa, codons in CODON_TABLE.items():
    for codon in codons:
        BINARIZE_NTS.append(f"{aa}_{codon}")   # e.g. "Phe_TTT"
for stop in STOP_CODONS:
    BINARIZE_NTS.append(f"Stop_{stop}")        # e.g. "Stop_TAA"

NON_TERMINALS = STRUCTURAL_NTS + AA_NAMES + HELPER_NTS + BINARIZE_NTS
NT_COUNT      = len(NON_TERMINALS)
S_COUNT       = NT_COUNT + T_COUNT   # total symbol alphabet

nt_to_idx = {nt: i  for i, nt in enumerate(NON_TERMINALS)}
t_to_idx  = {t:  i  for i, t  in enumerate(TERMINALS)}

print(f"Grammar: {NT_COUNT} non-terminals | {T_COUNT} terminals | "
      f"{_n_codons} sense codons | {len(BINARIZE_NTS)} binarization NTs")

def sym_idx(sym: str) -> int:
    if sym in nt_to_idx: return nt_to_idx[sym]
    if sym in t_to_idx:  return NT_COUNT + t_to_idx[sym]
    raise ValueError(f"Unknown symbol: '{sym}'")

def sym_name(idx: int) -> str:
    return NON_TERMINALS[idx] if idx < NT_COUNT else TERMINALS[idx - NT_COUNT]

def is_nt(sym: str) -> bool:
    return sym in nt_to_idx

# ══════════════════════════════════════════════════════════════════════
# §3  RULE POPULATION
# ══════════════════════════════════════════════════════════════════════
#
# ALL rules are strictly binary (CNF).
# Unary rules A → B are eliminated by inlining:
#   wherever A was a child in a binary rule, substitute B directly.
#
# Key inlinings applied:
#   RegulatoryRegion → Promoter   ⟹  Gene → Promoter CodingRegion
#   Exon → Protein                ⟹  CodingRegion → {Protein rules}
#   Locus → Gene|NonCoding        ⟹  Chromatid uses Gene/NonCoding directly
#   AminoAcid → {aa names}        ⟹  ProteinTail expanded for all 20 AAs

# Raw rule store: (lhs_str, r1_str, r2_str)
_RAW_RULES: list[tuple] = []

def R(lhs: str, r1: str, r2: str, w: float = 0.0):
    """Register one binary rule."""
    assert lhs in nt_to_idx,  f"Unknown LHS: {lhs}"
    assert sym_idx(r1) >= 0,  f"Unknown r1: {r1}"
    assert sym_idx(r2) >= 0,  f"Unknown r2: {r2}"
    _RAW_RULES.append((lhs, r1, r2, w))

# ── §3.1  Top-level structural rules ─────────────────────────────────

# Genome → Chromatid Chromatid | Chromatid GenomeR
# (Genome over multiple chromatids; single-chromatid genome uses Chromatid as root)
R("Genome", "Chromatid", "Chromatid")
R("Genome", "Chromatid", "GenomeR")
R("GenomeR", "Chromatid", "Chromatid")
R("GenomeR", "Chromatid", "GenomeR")

# Chromatid → Gene | NonCoding         (inlined Locus)
# Chromatid → Gene ChromatidR | NonCoding ChromatidR
R("Chromatid", "Gene",      "NonCoding")
R("Chromatid", "Gene",      "ChromatidR")
R("Chromatid", "NonCoding", "ChromatidR")
R("Chromatid", "NonCoding", "Gene")
R("ChromatidR", "Gene",     "ChromatidR")
R("ChromatidR", "NonCoding","ChromatidR")
R("ChromatidR", "Gene",     "NonCoding")
R("ChromatidR", "NonCoding","Gene")

# ── §3.2  Gene internal structure ─────────────────────────────────────
#
# Conceptual: Gene → RegulatoryRegion CodingRegion
# After inlining RegulatoryRegion → Promoter → CoreElement:
#
#   Gene → CoreElement  CodingRegion
#   Gene → Promoter     CodingRegion   (Promoter = CoreElement TFBS)
#   Gene → Enhancer     CodingRegion   (Enhancer before coding)

R("Gene", "CoreElement", "CodingRegion")
R("Gene", "Promoter",    "CodingRegion")
R("Gene", "Enhancer",    "CodingRegion")

# Promoter → CoreElement TFBS
R("Promoter", "CoreElement", "TFBS")

# CoreElement: 2-base consensus sequences  (TATA-like)
# All 2-mer combinations; real TATA box = T,A pattern
for b1 in TERMINALS:
    for b2 in TERMINALS:
        R("CoreElement", b1, b2)

# TFBS: 2+ bases (transcription factor binding motif, 6–20 bp in biology)
# Here we model 2-base minimum for tractability
for b1 in TERMINALS:
    for b2 in TERMINALS:
        R("TFBS", b1, b2)
    R("TFBS", b1, "TFBSBase")         # 3+ base TFBS

for b1 in TERMINALS:
    for b2 in TERMINALS:
        R("TFBSBase", b1, b2)
    R("TFBSBase", b1, "TFBSBase")

# Enhancer → TFBS TFBS | TFBS Enhancer
R("Enhancer", "TFBS", "TFBS")
R("Enhancer", "TFBS", "Enhancer")

# ── §3.3  CodingRegion ────────────────────────────────────────────────
#
# Conceptual: CodingRegion → Exon (Intron Exon)*
# After inlining Exon → Protein:
#
#   CodingRegion → Protein           (single exon)
#   CodingRegion → Protein CRTail    (multiple exons)
#
# "Protein" as direct child of CodingRegion is unary — inline further:
# Copy all Protein-generating rules with CodingRegion as LHS.
# (Done at end of §3.4 once Protein rules are defined.)

R("CodingRegion", "Protein", "CRTail")

R("CRTail", "IntronProt", "CRTail")
R("CRTail", "IntronProt", "Protein")

# IntronProt → Intron Protein  (inlined Intron Exon → Intron Protein)
R("IntronProt", "Intron", "Protein")

# ── §3.4  Intron and NonCoding  (Base+) ───────────────────────────────

for b in TERMINALS:
    for b2 in TERMINALS:
        R("Intron",   b,  b2)
        R("NonCoding", b, b2)
    R("Intron",    b, "InBase")
    R("NonCoding", b, "NCBase")

for b in TERMINALS:
    for b2 in TERMINALS:
        R("InBase",  b, b2)
        R("NCBase",  b, b2)
    R("InBase",  b, "InBase")
    R("NCBase",  b, "NCBase")

# ── §3.5  Protein and ProteinTail ─────────────────────────────────────
#
# Conceptual:
#   Protein     → StartAA AminoAcid* StopSignal
#   StartAA     → Met          (all proteins start with Met)
#   AminoAcid   → any of 20 AAs
#
# After inlining StartAA → Met and AminoAcid → {each AA}:
#
#   Protein     → Met StopSignal       (Met-Stop: minimal)
#   Protein     → Met ProteinTail      (Met + body)
#   ProteinTail → <aa> StopSignal      (last residue before stop)
#   ProteinTail → <aa> ProteinTail     (recursive body)
#
# One rule per amino acid × {StopSignal, ProteinTail} = 40 rules.

R("Protein", "Met", "StopSignal")    # minimal: Met-Stop
R("Protein", "Met", "ProteinTail")   # Met + body

for aa in AA_NAMES:
    R("ProteinTail", aa, "StopSignal")
    R("ProteinTail", aa, "ProteinTail")

# ── Inline CodingRegion → Protein ─────────────────────────────────────
# (copies of all Protein top-level rules with CodingRegion as LHS)
R("CodingRegion", "Met", "StopSignal")
R("CodingRegion", "Met", "ProteinTail")

# ── §3.6  Genetic code: each amino acid → codon → bases ──────────────
#
# For codon B1B2B3 encoding amino acid AA:
#   AA         → B1  AA_B1B2B3    (binary rule)
#   AA_B1B2B3  → B2  B3           (binarization NT)
#
# This preserves CFG property: each rule has exactly two RHS symbols.

for aa, codons in CODON_TABLE.items():
    for codon in codons:
        b1, b2, b3 = codon[0], codon[1], codon[2]
        inter = f"{aa}_{codon}"          # e.g. "Phe_TTT"
        R(aa,    b1,  inter)             # Phe  → T  Phe_TTT
        R(inter, b2,  b3)               # Phe_TTT → T  T

# ── §3.7  Stop codons (binarized) ─────────────────────────────────────
for codon in STOP_CODONS:
    b1, b2, b3 = codon[0], codon[1], codon[2]
    inter = f"Stop_{codon}"             # e.g. "Stop_TAA"
    R("StopSignal", b1, inter)          # StopSignal → T  Stop_TAA
    R(inter,        b2, b3)             # Stop_TAA   → A  A

print(f"Rules registered: {len(_RAW_RULES)}")

# ══════════════════════════════════════════════════════════════════════
# §4  BUILD RULE INDEX  (fast lookup for CYK)
# ══════════════════════════════════════════════════════════════════════

# rule_index[lhs_idx] = list of (r1_idx, r2_idx)
rule_index: dict[int, list[tuple[int,int]]] = defaultdict(list)
for lhs, r1, r2, _w in _RAW_RULES:
    rule_index[nt_to_idx[lhs]].append((sym_idx(r1), sym_idx(r2)))

# reverse_index[(r1_idx, r2_idx)] = list of lhs_idx
# Used for bottom-up CYK
reverse_index: dict[tuple, list[int]] = defaultdict(list)
for lhs, r1, r2, _w in _RAW_RULES:
    key = (sym_idx(r1), sym_idx(r2))
    reverse_index[key].append(nt_to_idx[lhs])

print(f"Unique (r1,r2) pairs: {len(reverse_index)}")

# ══════════════════════════════════════════════════════════════════════
# §5  PURE PYTHON CYK PARSER
# ══════════════════════════════════════════════════════════════════════

def cyk_parse(seq: str, root_sym: str) -> tuple:
    """
    CYK bottom-up parser.

    Returns:
        (success: bool, chart: list, back: list)

    chart[i][j] = frozenset of symbol indices spanning positions [i,j)
    back[i][j][sym_idx] = (k, r1_idx, r2_idx) — for tree reconstruction
    """
    N = len(seq)
    # chart[i][j]: set of symbol indices covering span [i,j)
    chart = [[set() for _ in range(N + 1)] for _ in range(N)]
    back  = [[dict() for _ in range(N + 1)] for _ in range(N)]

    # ── Fill length-1 spans (terminals) ──────────────────────────────
    for i, ch in enumerate(seq):
        if ch not in t_to_idx:
            raise ValueError(f"Character '{ch}' at position {i} is not in {TERMINALS}")
        t_sym = NT_COUNT + t_to_idx[ch]   # terminal symbol index
        chart[i][i+1].add(t_sym)

    # ── Fill length >= 2 (bottom-up) ─────────────────────────────────
    for length in range(2, N + 1):
        for i in range(N - length + 1):
            j = i + length
            for k in range(i + 1, j):
                left  = chart[i][k]
                right = chart[k][j]
                for r1 in left:
                    for r2 in right:
                        for lhs_i in reverse_index.get((r1, r2), []):
                            if lhs_i not in back[i][j]:
                                chart[i][j].add(lhs_i)
                                back[i][j][lhs_i] = (k, r1, r2)

    root_i  = nt_to_idx[root_sym]
    success = root_i in chart[0][N]
    return success, chart, back

# ══════════════════════════════════════════════════════════════════════
# §6  TREE RECONSTRUCTION
# ══════════════════════════════════════════════════════════════════════

def reconstruct_tree(seq: str, back: list, root_sym: str) -> dict:
    """
    Walk the CYK backpointer chart to build a parse tree dict.
    Node format: {"name": str, "span": "i:j", "children": [...]}
    """
    N      = len(seq)
    root_i = nt_to_idx[root_sym]

    def build(i: int, j: int, sym: int) -> dict:
        label = sym_name(sym)
        node  = {"name": label, "span": f"{i}:{j}", "children": []}

        if j == i + 1:
            # Single-position span
            if sym >= NT_COUNT:
                # This IS the terminal character
                node["name"] = seq[i]
                node["children"] = []
            else:
                # NT at a single character — show the base as leaf
                node["children"] = [
                    {"name": seq[i], "span": str(i), "children": []}
                ]
            return node

        if sym not in back[i][j]:
            # No backpointer found (shouldn't happen if parse succeeded)
            node["children"] = [{"name": f"[{seq[i:j]}]", "children": []}]
            return node

        k, r1, r2 = back[i][j][sym]
        node["children"] = [build(i, k, r1), build(k, j, r2)]
        return node

    return build(0, N, root_i)

# ══════════════════════════════════════════════════════════════════════
# §7  OPTIONAL: torch_struct SCORING
# ══════════════════════════════════════════════════════════════════════

def build_torch_rules():
    """Build the rules tensor for torch_struct SentCFG."""
    import torch
    r = torch.full((1, NT_COUNT, S_COUNT, S_COUNT), NEG_INF)
    for lhs, r1s, r2s, w in _RAW_RULES:
        r[0, nt_to_idx[lhs], sym_idx(r1s), sym_idx(r2s)] = w
    return r

def torch_log_partition(seq: str, root_sym: str):
    """
    Compute log-partition (inside score) via torch_struct.
    Returns log Z (float).  Useful for learning and scoring.
    """
    import torch
    from torch_struct import SentCFG

    N = len(seq)
    terms_t = torch.full((1, N, T_COUNT), NEG_INF)
    for i, ch in enumerate(seq):
        terms_t[0, i, t_to_idx[ch]] = 0.0

    root_v = torch.full((1, NT_COUNT), NEG_INF)
    root_v[0, nt_to_idx[root_sym]] = 0.0

    rules_t = build_torch_rules()
    dist    = SentCFG((terms_t, rules_t, root_v), lengths=torch.tensor([N]))
    return dist.log_partition.item()

# ══════════════════════════════════════════════════════════════════════
# §8  VISUALIZATION
# ══════════════════════════════════════════════════════════════════════

# Colour palette matching the LaTeX document (bg, text)
_PALETTE = {
    # Genome hierarchy
    "Genome":       ("#1e3a5f", "#bfdbfe"),
    "Chromatid":    ("#1e3a5f", "#bfdbfe"),
    "Gene":         ("#0ea5e9", "#e0f2fe"),
    # Regulatory
    "Promoter":     ("#ec4899", "#fce7f3"),
    "CoreElement":  ("#f59e0b", "#1c1917"),
    "Enhancer":     ("#ec4899", "#fce7f3"),
    "TFBS":         ("#f59e0b", "#1c1917"),
    # Coding
    "CodingRegion": ("#10b981", "#d1fae5"),
    "Intron":       ("#64748b", "#f1f5f9"),
    "NonCoding":    ("#94a3b8", "#1e293b"),
    # Protein
    "Protein":      ("#ef4444", "#fee2e2"),
    "ProteinTail":  ("#f97316", "#ffedd5"),
    "StopSignal":   ("#7f1d1d", "#fecaca"),
}
for _aa in AA_NAMES:
    _PALETTE[_aa] = ("#92400e", "#fef3c7")   # amber for amino acids

def _node_colour(label: str) -> tuple[str, str]:
    """Return (bg, fg) hex colours for a node label."""
    if label in _PALETTE:
        bg, fg = _PALETTE[label]
        return bg, fg
    if label in TERMINALS:
        return "#065f46", "#d1fae5"   # dark emerald for terminals
    if label.endswith(tuple(STOP_CODONS)) or label.startswith("Stop_"):
        return "#7f1d1d", "#fecaca"
    # Binarization NTs (AA_codon)
    for aa in AA_NAMES:
        if label.startswith(aa + "_"):
            return "#78350f", "#fef9c3"
    return "#334155", "#f8fafc"       # default slate


def ascii_tree(node: dict, prefix: str = "", last: bool = True):
    """Print a parse tree in console-friendly format."""
    marker = "└── " if last else "├── "
    span   = f"  [{node.get('span', '')}]" if node.get("span") else ""
    print(prefix + marker + node["name"] + span)
    children = node.get("children", [])
    extension = "    " if last else "│   "
    for i, child in enumerate(children):
        ascii_tree(child, prefix + extension, i == len(children) - 1)


def graphviz_tree(root_node: dict, filename: str = "tree"):
    """Render parse tree to a PNG using graphviz."""
    dot = graphviz.Digraph(comment="Genomic Parse Tree")
    dot.attr(rankdir="TB", nodesep="0.25", ranksep="0.4",
             bgcolor="#0f172a")
    dot.attr("node",
             shape="box",
             style="filled,rounded",
             fontname="Helvetica Neue",
             fontsize="10",
             margin="0.12,0.06")
    dot.attr("edge", color="#475569", arrowsize="0.6")

    counter = [0]

    def add(parent_id: str | None, node: dict):
        nid = str(counter[0]); counter[0] += 1
        bg, fg = _node_colour(node["name"])
        label  = node["name"]
        dot.node(nid, label, fillcolor=bg, fontcolor=fg,
                 color=bg, penwidth="1.0")
        if parent_id is not None:
            dot.edge(parent_id, nid)
        for child in node.get("children", []):
            add(nid, child)

    add(None, root_node)
    out = dot.render(filename, format="png", cleanup=True)
    print(f"    → PNG saved: {out}")
    return out


# ══════════════════════════════════════════════════════════════════════
# §9  MUTATION ANALYSIS
# ══════════════════════════════════════════════════════════════════════

def classify_mutation(seq_wt: str, pos: int, new_base: str, root_sym: str = "Protein"):
    """
    Classify a point mutation at `pos` in sequence `seq_wt`.

    Returns a dict with:
      mutation_type:  'synonymous' | 'nonsynonymous' | 'nonsense' | 'promoter' | 'other'
      wt_codon:       wild-type codon string (if in coding region)
      mut_codon:      mutated codon string
      wt_aa:          wild-type amino acid (if in CDS)
      mut_aa:         mutated amino acid (or 'Stop')
      explanation:    plain-text description
    """
    seq_mt = seq_wt[:pos] + new_base + seq_wt[pos+1:]

    ok_wt, _, back_wt = cyk_parse(seq_wt, root_sym)
    ok_mt, _, back_mt = cyk_parse(seq_mt, root_sym)

    result = {
        "wt_seq":   seq_wt,
        "mut_seq":  seq_mt,
        "position": pos,
        "wt_base":  seq_wt[pos],
        "new_base": new_base,
        "wt_parse": ok_wt,
        "mt_parse": ok_mt,
    }

    # Determine which codon position was affected
    # (pos relative to start of protein coding region)
    codon_pos = pos // 3
    within_codon = pos % 3
    wt_codon  = seq_wt[codon_pos*3 : codon_pos*3+3]
    mut_codon = seq_mt[codon_pos*3 : codon_pos*3+3]
    result["wt_codon"]  = wt_codon
    result["mut_codon"] = mut_codon

    # Look up amino acids
    def codon_to_aa(codon):
        if codon in STOP_CODONS: return "Stop"
        for aa, codons in CODON_TABLE.items():
            if codon in codons: return aa
        return "?"

    wt_aa  = codon_to_aa(wt_codon)
    mut_aa = codon_to_aa(mut_codon)
    result["wt_aa"]  = wt_aa
    result["mut_aa"] = mut_aa

    if wt_aa == mut_aa:
        result["mutation_type"] = "synonymous"
        result["explanation"]   = (
            f"Synonymous (silent): {wt_codon}→{mut_codon}, "
            f"both encode {wt_aa}.  "
            f"Parse-tree node '{wt_aa}' unchanged; only terminal string beneath it changes."
        )
    elif mut_aa == "Stop":
        result["mutation_type"] = "nonsense"
        result["explanation"]   = (
            f"Nonsense (premature stop): {wt_codon}({wt_aa})→{mut_codon}(Stop).  "
            f"ProteinTail derivation terminates early — protein truncated."
        )
    elif wt_aa == "?":
        result["mutation_type"] = "other"
        result["explanation"]   = f"Position outside coding region."
    else:
        result["mutation_type"] = "nonsynonymous"
        result["explanation"]   = (
            f"Non-synonymous (missense): {wt_codon}({wt_aa})→{mut_codon}({mut_aa}).  "
            f"Non-terminal node label changes from '{wt_aa}' to '{mut_aa}'."
        )

    return result


# ══════════════════════════════════════════════════════════════════════
# §10  TEST SUITE
# ══════════════════════════════════════════════════════════════════════

# Each entry: (sequence, root_symbol, description)
TEST_CASES = [
    # ── Protein-level tests ──────────────────────────────────────────
    ("ATGTAA",           "Protein",
     "Met · Stop  [minimal 2-codon protein]"),

    ("ATGTTTTAA",        "Protein",
     "Met-Phe · Stop"),

    ("ATGTTTGCCTAA",     "Protein",
     "Met-Phe-Ala · Stop"),

    ("ATGCCTGCTTAA",     "Protein",
     "Met-Pro-Ala · Stop"),

    ("ATGTTTCCTGCTTAA",  "Protein",
     "Met-Phe-Pro-Ala · Stop  [4 residues]"),

    # ── Gene-level tests (2-base CoreElement + coding region) ────────
    ("AAATGTTTTAA",      "Gene",
     "CoreElem(AA) · Gene: Met-Phe · Stop"),

    ("TAATGCCTGCTTAA",   "Gene",
     "CoreElem(TA) · Gene: Met-Pro-Ala · Stop"),

    # ── Mutation analysis demo ────────────────────────────────────────
    # (handled separately below)
]

DIVIDER = "═" * 62

def run_tests():
    print(f"\n{DIVIDER}")
    print("  GENOMIC CFG PARSER — TEST SUITE")
    print(DIVIDER)

    for seq, root, desc in TEST_CASES:
        print(f"\n  Seq : {seq}  (n={len(seq)})")
        print(f"  Root: {root}")
        print(f"  Desc: {desc}")

        # CYK parse
        ok, chart, back = cyk_parse(seq, root)
        status = "✓  parse succeeded" if ok else "✗  NO VALID PARSE"
        print(f"  CYK : {status}")

        if ok:
            tree = reconstruct_tree(seq, back, root)
            print()
            ascii_tree(tree)

            if HAS_GV:
                safe = seq + "_" + root
                graphviz_tree(tree, filename=safe)

            if HAS_TORCH:
                try:
                    lz = torch_log_partition(seq, root)
                    print(f"\n  log Z (torch_struct) = {lz:.4f}")
                except Exception as e:
                    print(f"\n  [torch_struct error: {e}]")
        print()

    # ── Mutation analysis ─────────────────────────────────────────────
    print(DIVIDER)
    print("  MUTATION ANALYSIS DEMO")
    print(DIVIDER)

    wt = "ATGTTTGCCTAA"   # Met-Phe-Ala-Stop

    mutations = [
        # (position, new_base, comment)
        (3,  "C",  "TTT→TCT  Phe→Ser  [nonsynonymous]"),
        (5,  "C",  "TTT→TTC  Phe→Phe  [synonymous — silent]"),
        (3,  "A",  "TTT→TAT  Phe→Tyr  [nonsynonymous]"),
        (3,  "G",  "TTT→TGT  Phe→Cys  [nonsynonymous]"),
        (4,  "A",  "TTT→TAT  first T of Phe → nonsense?"),
        # introduce early stop
        (6,  "G",  "GCT→GGT  Ala→Gly  [nonsynonymous]"),
        (6,  "A",  "GCT→ACT  Ala→Thr  [nonsynonymous]"),
    ]

    print(f"\n  Wild-type: {wt}  (Met-Phe-Ala-Stop)")
    for pos, nb, comment in mutations:
        r = classify_mutation(wt, pos, nb, root_sym="Protein")
        print(f"\n  pos={pos}  {r['wt_base']}→{nb}  |  {comment}")
        print(f"  Codon: {r['wt_codon']}({r['wt_aa']}) → {r['mut_codon']}({r['mut_aa']})")
        print(f"  Type : {r['mutation_type'].upper()}")
        print(f"  Note : {r['explanation']}")


def print_grammar_summary():
    print(f"\n{DIVIDER}")
    print("  GRAMMAR SUMMARY")
    print(DIVIDER)
    print(f"\n  {'Category':<25} {'Count':>6}")
    print("  " + "─"*35)
    print(f"  {'Structural NTs':<25} {len(STRUCTURAL_NTS):>6}")
    print(f"  {'Amino acid NTs':<25} {len(AA_NAMES):>6}")
    print(f"  {'Helper NTs':<25} {len(HELPER_NTS):>6}")
    print(f"  {'Binarization NTs':<25} {len(BINARIZE_NTS):>6}")
    print(f"  {'─'*30}")
    print(f"  {'Total NTs':<25} {NT_COUNT:>6}")
    print(f"  {'Terminals':<25} {T_COUNT:>6}")
    print(f"  {'Total binary rules':<25} {len(_RAW_RULES):>6}")
    print()

    print("  Degeneracy of genetic code (encoded as rule count):")
    for aa, codons in sorted(CODON_TABLE.items(), key=lambda x: len(x[1])):
        bar = "█" * len(codons)
        print(f"    {aa:<4}  {bar:<6} {len(codons)} codon(s):  {', '.join(codons)}")
    print(f"    Stop  {'█'*3}   3 codons:  {', '.join(STOP_CODONS)}")
    print()


if __name__ == "__main__":
    print_grammar_summary()
    run_tests()


(dot.exe:8288): Pango-WARNING **: couldn't load font "Helvetica Neue Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.


Grammar: 104 non-terminals | 4 terminals | 61 sense codons | 64 binarization NTs
Rules registered: 330
Unique (r1,r2) pairs: 148

══════════════════════════════════════════════════════════════
  GRAMMAR SUMMARY
══════════════════════════════════════════════════════════════

  Category                   Count
  ───────────────────────────────────
  Structural NTs                12
  Amino acid NTs                20
  Helper NTs                     8
  Binarization NTs              64
  ──────────────────────────────
  Total NTs                    104
  Terminals                      4
  Total binary rules           330

  Degeneracy of genetic code (encoded as rule count):
    Met   █      1 codon(s):  ATG
    Trp   █      1 codon(s):  TGG
    Phe   ██     2 codon(s):  TTT, TTC
    Tyr   ██     2 codon(s):  TAT, TAC
    His   ██     2 codon(s):  CAT, CAC
    Gln   ██     2 codon(s):  CAA, CAG
    Asn   ██     2 codon(s):  AAT, AAC
    Lys   ██     2 codon(s):  AAA, AAG
    Asp   ██     


(dot.exe:19268): Pango-WARNING **: couldn't load font "Helvetica Neue Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

(dot.exe:22988): Pango-WARNING **: couldn't load font "Helvetica Neue Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

(dot.exe:22396): Pango-WARNING **: couldn't load font "Helvetica Neue Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.


    → PNG saved: ATGTTTTAA_Protein.png

  [torch_struct error: 'SentCFG' object has no attribute 'log_partition']


  Seq : ATGTTTGCCTAA  (n=12)
  Root: Protein
  Desc: Met-Phe-Ala · Stop
  CYK : ✓  parse succeeded

└── Protein  [0:12]
    ├── Met  [0:3]
    │   ├── A  [0:1]
    │   └── Met_ATG  [1:3]
    │       ├── T  [1:2]
    │       └── G  [2:3]
    └── ProteinTail  [3:12]
        ├── Phe  [3:6]
        │   ├── T  [3:4]
        │   └── Phe_TTT  [4:6]
        │       ├── T  [4:5]
        │       └── T  [5:6]
        └── ProteinTail  [6:12]
            ├── Ala  [6:9]
            │   ├── G  [6:7]
            │   └── Ala_GCC  [7:9]
            │       ├── C  [7:8]
            │       └── C  [8:9]
            └── StopSignal  [9:12]
                ├── T  [9:10]
                └── Stop_TAA  [10:12]
                    ├── A  [10:11]
                    └── A  [11:12]
    → PNG saved: ATGTTTGCCTAA_Protein.png

  [torch_struct error: 'SentCFG' object has no attribute 'log_partition']


 


(dot.exe:21680): Pango-WARNING **: couldn't load font "Helvetica Neue Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

(dot.exe:30060): Pango-WARNING **: couldn't load font "Helvetica Neue Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.


    → PNG saved: ATGTTTCCTGCTTAA_Protein.png

  [torch_struct error: 'SentCFG' object has no attribute 'log_partition']


  Seq : AAATGTTTTAA  (n=11)
  Root: Gene
  Desc: CoreElem(AA) · Gene: Met-Phe · Stop
  CYK : ✓  parse succeeded

└── Gene  [0:11]
    ├── CoreElement  [0:2]
    │   ├── A  [0:1]
    │   └── A  [1:2]
    └── CodingRegion  [2:11]
        ├── Met  [2:5]
        │   ├── A  [2:3]
        │   └── Met_ATG  [3:5]
        │       ├── T  [3:4]
        │       └── G  [4:5]
        └── ProteinTail  [5:11]
            ├── Phe  [5:8]
            │   ├── T  [5:6]
            │   └── Phe_TTT  [6:8]
            │       ├── T  [6:7]
            │       └── T  [7:8]
            └── StopSignal  [8:11]
                ├── T  [8:9]
                └── Stop_TAA  [9:11]
                    ├── A  [9:10]
                    └── A  [10:11]
    → PNG saved: AAATGTTTTAA_Gene.png

  [torch_struct error: 'SentCFG' object has no attribute 'log_partition']


  Seq : TAATGCCTGCTTAA  (n=14)
  Root: 


(dot.exe:23884): Pango-WARNING **: couldn't load font "Helvetica Neue Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.


    → PNG saved: TAATGCCTGCTTAA_Gene.png

  [torch_struct error: 'SentCFG' object has no attribute 'log_partition']

══════════════════════════════════════════════════════════════
  MUTATION ANALYSIS DEMO
══════════════════════════════════════════════════════════════

  Wild-type: ATGTTTGCCTAA  (Met-Phe-Ala-Stop)

  pos=3  T→C  |  TTT→TCT  Phe→Ser  [nonsynonymous]
  Codon: TTT(Phe) → CTT(Leu)
  Type : NONSYNONYMOUS
  Note : Non-synonymous (missense): TTT(Phe)→CTT(Leu).  Non-terminal node label changes from 'Phe' to 'Leu'.

  pos=5  T→C  |  TTT→TTC  Phe→Phe  [synonymous — silent]
  Codon: TTT(Phe) → TTC(Phe)
  Type : SYNONYMOUS
  Note : Synonymous (silent): TTT→TTC, both encode Phe.  Parse-tree node 'Phe' unchanged; only terminal string beneath it changes.

  pos=3  T→A  |  TTT→TAT  Phe→Tyr  [nonsynonymous]
  Codon: TTT(Phe) → ATT(Ile)
  Type : NONSYNONYMOUS
  Note : Non-synonymous (missense): TTT(Phe)→ATT(Ile).  Non-terminal node label changes from 'Phe' to 'Ile'.

  pos=3  T→G  |  TTT